### Optical Network Topology & End-to-End Path QoT Estimation

This tutorial demonstrates how to perform mesh optical network QoT estimation using the Closed-Form Model (CFM) framework:
* **Network Topology:** Parsing cost matrices into graph models and calculating $k$-shortest paths ($k$SP).
* **Multi-Band Grid:** Dynamic allocation across L, C, S, and E bands with guard bands.
* **Traffic & Routing:** Simulating realistic channel loading, modulation assignments, and launch power allocations.
* **Accumulated Path QoT:** Propagating signals span-by-span, accumulating inverse GSNR along transparent optical lightpaths, and applying ROADM/WSS node penalties.
* **Post-Processing:** Visualizing route-level optical performance and modulation allocations.


In [1]:
import sys
from pathlib import Path
import warnings
import time
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from scipy.io import loadmat

warnings.filterwarnings("ignore")

# Setup project source imports relative to workspace root
base_dir = Path.cwd()
src_dir = base_dir.parent / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from CFM.core.network import Link, LinkParameters, Topology
from CFM.core.band import Band, OpticalParameters
from CFM.core.qot_estimator import (
    ParameterBuilder, 
    ISRSSolver, 
    NLISolver, 
    ASESolver, 
    OSNRCalculator
)
from CFM.core.post_process import ModulationConnectionPlotter
from CFM.utils.build_alpha_db import build_alpha_for_band, get_data_file_path
from CFM.utils.connection_profile import build_connection_profile

# Define output directory
results_dir = base_dir.parent / "results" / "topology"
results_dir.mkdir(parents=True, exist_ok=True)

### 1. Multi-Band Spectrum & Guard Band Grid Setup

Construct an ultra-wideband transmission window (L, C, S, and E bands) including guard band channels between adjacent bands to prevent edge filtering distortions.


In [3]:
def setup_multiband_grid(band_config="LC", symbol_rate_baud=120e9, channel_spacing_ghz=150.0, n_guard_channels=3):
    """
    Construct continuous multi-band optical frequency grids with physical guard bands.
    """
    c = 299792458
    channel_spacing_thz = channel_spacing_ghz * 1e9 / 1e12
    band_params = OpticalParameters(Rs_mat=symbol_rate_baud)

    # L-Band (1575.4 nm to 1626 nm)
    band_l = Band("L", (c / 1626e-9) / 1e12, (c / 1575.4e-9) / 1e12, band_params, channel_spacing_thz)
    spec_l = band_l.calc_spectrum() + channel_spacing_thz * 0.5

    # Guard Band: L to C
    gband_l_c = Band("G", spec_l[0] + channel_spacing_thz * 0.5, spec_l[0] + n_guard_channels * channel_spacing_thz, band_params, channel_spacing_thz)
    guard_l_c = spec_l[0] + np.arange(1, n_guard_channels + 1) * channel_spacing_thz

    # C-Band (1524.1 nm to 1575.4 nm)
    band_c = Band("C", guard_l_c[-1] + channel_spacing_thz * 0.5, (c / 1524.1e-9) / 1e12, band_params, channel_spacing_thz)
    spec_c = band_c.calc_spectrum() + channel_spacing_thz * 0.5

    spectra_list = [spec_l[::-1], gband_l_c.calc_spectrum() + channel_spacing_thz * 0.5, spec_c[::-1]]
    bands = [band_l, gband_l_c, band_c]

    # S-Band (1460.0 nm to 1524.1 nm)
    if "S" in band_config:
        gband_c_s = Band("G", spec_c[0] + channel_spacing_thz * 0.5, spec_c[0] + n_guard_channels * channel_spacing_thz, band_params, channel_spacing_thz)
        guard_c_s = spec_c[0] + np.arange(1, n_guard_channels + 1) * channel_spacing_thz
        band_s = Band("S", guard_c_s[-1] + channel_spacing_thz * 0.5, (c / 1460.0e-9) / 1e12, band_params, channel_spacing_thz)
        spec_s = band_s.calc_spectrum() + channel_spacing_thz * 0.5

        spectra_list.extend([gband_c_s.calc_spectrum() + channel_spacing_thz * 0.5, spec_s[::-1]])
        bands.extend([gband_c_s, band_s])

    # E-Band (1360.0 nm to 1460.0 nm)
    if "E" in band_config:
        gband_s_e = Band("G", spec_s[0] + channel_spacing_thz * 0.5, spec_s[0] + n_guard_channels * channel_spacing_thz, band_params, channel_spacing_thz)
        guard_s_e = spec_s[0] + np.arange(1, n_guard_channels + 1) * channel_spacing_thz
        band_e = Band("E", guard_s_e[-1] + channel_spacing_thz * 0.5, (c / 1360.0e-9) / 1e12, band_params, channel_spacing_thz)
        spec_e = band_e.calc_spectrum() + channel_spacing_thz * 0.5

        spectra_list.extend([gband_s_e.calc_spectrum() + channel_spacing_thz * 0.5, spec_e[::-1]])
        bands.extend([gband_s_e, band_e])

    grid_center = np.concatenate(spectra_list) * 1e12
    alpha_dB_dynamic = np.concatenate([np.squeeze(build_alpha_for_band(None, None, b)) for b in bands])

    return grid_center, bands, alpha_dB_dynamic

# Initialize configuration (e.g., C + L + S)
grid_center, bands, alpha_dB_dynamic = setup_multiband_grid(band_config="LCS")
n_channels = len(grid_center)

print(f"Spectrum initialized with {n_channels} total channels across {[b.name for b in bands if b.name != 'G']} active bands.")

Spectrum initialized with 140 total channels across ['L', 'C', 'S'] active bands.


### 2. Network Topology & Routing Profiles

In [4]:
# Load network adjacency matrix
net_matrix_path = get_data_file_path("JPN4812_netCostMatrix.mat")
mat_data = loadmat(str(net_matrix_path))
net_cost_matrix = mat_data["netCostMatrix"]

# Instantiate Topology
topology = Topology(netcost_matrix=net_cost_matrix, link_params=LinkParameters(), span_length=80.0)
G = topology.get_graph()

# Core node set for demand endpoints
core_nodes = np.array([7, 8, 9, 11, 19, 23, 25, 26, 32, 35, 36, 40])
k_paths = 3
span_length_target = 80.0

# Generate routing profiles
G, all_connections_profile, node_degrees = build_connection_profile(
    G=G.copy(), 
    core_nodes=core_nodes, 
    kSP=k_paths, 
    Lspan=span_length_target
)

num_connections = all_connections_profile.shape[0]
print(f"Topology Graph: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Physical Fiber Links.")
print(f"Generated {num_connections} Demands with {k_paths} Shortest Paths each.")


Topology Graph: 48 Nodes, 82 Physical Fiber Links.
Generated 66 Demands with 3 Shortest Paths each.


### 3. Dynamic Traffic & Modulation Assignment

In [ ]:
def generate_traffic(n_channels, total_distance_km, activity_factor=0.8, p_in_base_dbm=0.0):
    """
    Generate channel launch powers, activity masks, and modulation format levels.
    """
    p_in_array = np.full(n_channels, -80.0)
    active_indices = np.where(np.random.rand(n_channels) < activity_factor)[0]
    p_in_array[active_indices] = p_in_base_dbm

    # Distance-adaptive modulation allocation
    if total_distance_km <= 600:
        mod_val = 6
    elif total_distance_km <= 1200:
        mod_val = 5
    elif total_distance_km <= 2000:
        mod_val = 4
    elif total_distance_km <= 3000:
        mod_val = 3
    elif total_distance_km <= 5000:
        mod_val = 2
    else:
        mod_val = 1

    mod_levels = np.full(n_channels, mod_val)
    
    phi_mapping = np.array([1.0, 1.0, 2/3, 17/25, 0.69, 13/21])
    phi_array = phi_mapping[np.clip(mod_levels - 1, 0, 5)]

    return p_in_array, mod_levels, phi_array, active_indices

### 4. End-to-End Lightpath QoT Simulation

In [7]:
wss_penalty_degree = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] # zero for the simulation
max_connections_to_evaluate = min(10, num_connections)

results_records = []
modulation_connection_matrix = np.zeros((max_connections_to_evaluate, n_channels, k_paths), dtype=int)

t_start = time.perf_counter()

for conn_idx in range(max_connections_to_evaluate):
    src = all_connections_profile[conn_idx, 0]
    dst = all_connections_profile[conn_idx, 1]
    
    # Evaluate the primary shortest path (k=0)
    path_nodes = all_connections_profile[conn_idx, 5][0]
    total_length_km = all_connections_profile[conn_idx, 3][0]
    num_amps_inline = all_connections_profile[conn_idx, 6][0]

    # Deconstruct path into constituent spans
    all_path_spans = []
    node_penalties = []
    current_span_idx = 0

    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        edge_weight = G[u][v]["weight"]
        n_spans_link = int(np.ceil(edge_weight / span_length_target))
        span_length_actual = edge_weight / n_spans_link

        for s in range(n_spans_link):
            all_path_spans.append(span_length_actual)
            if s == n_spans_link - 1 and i < len(path_nodes) - 2:
                degree_idx = min(int(node_degrees[int(v) - 1]) - 1, len(wss_penalty_degree) - 1)
                node_penalties.append((current_span_idx, 10 ** (wss_penalty_degree[degree_idx] / 10)))
            current_span_idx += 1

    p_in_arr, mod_levels, phi_arr, active_idx = generate_traffic(n_channels, total_length_km, activity_factor=0.7)
    modulation_connection_matrix[conn_idx, :, 0] = mod_levels

    inv_gsnr_accum = np.zeros(len(active_idx))

    for s_idx, l_span in enumerate(all_path_spans):
        span_link = Link(
            name=f"c{conn_idx}_s{s_idx}", 
            length=[l_span], 
            num_span=1, 
            num_amp=1, 
            link_params=LinkParameters()
        )

        span_params = ParameterBuilder(
            link=span_link,
            bands=bands,
            P_in=p_in_arr,
            grid_center=grid_center,
            P_in_is_tx_power=True,
            PHI=phi_arr,
            alpha_dB_LCS=alpha_dB_dynamic.reshape(1, -1)
        )

        _, p_out_dbm, a0, a1, sig = ISRSSolver(span_params, model="FLP").solve()
        ase_power = ASESolver(span_params).solve()
        nli_power = NLISolver(span_params, a0, a1, sig).solve()

        _, _, gsnr_total_db = OSNRCalculator(span_params, ase_power, nli_power).compute()
        
        span_gsnr_linear = 10 ** (-gsnr_total_db[-1].flatten()[active_idx] / 10)
        inv_gsnr_accum += span_gsnr_linear

        for target_span, penalty_factor in node_penalties:
            if s_idx == target_span:
                inv_gsnr_accum *= penalty_factor

    transceiver_snr_limit_db = 36.0
    filtering_penalty_db = 1.0 + 0.05 * num_amps_inline
    final_path_gsnr_db = -10 * np.log10(inv_gsnr_accum + 10 ** (-transceiver_snr_limit_db / 10)) - filtering_penalty_db

    results_records.append({
        "Connection": f"Node {src} -> Node {dst}",
        "Distance (km)": total_length_km,
        "Spans": len(all_path_spans),
        "Active Channels": len(active_idx),
        "Allocated Modulation": f"{mod_levels[active_idx[0]]}-QAM",
        "Mean GSNR (dB)": np.mean(final_path_gsnr_db),
        "Min GSNR (dB)": np.min(final_path_gsnr_db),
        "Max GSNR (dB)": np.max(final_path_gsnr_db)
    })

t_elapsed = time.perf_counter() - t_start
print(f"Completed physical-layer simulation for {max_connections_to_evaluate} lightpaths in {t_elapsed:.2f} s.")

df_results = pd.DataFrame(results_records)
display(df_results)



Completed physical-layer simulation for 10 lightpaths in 67.01 s.


,Connection,Distance (km),Spans,Active Channels,Allocated Modulation,Mean GSNR (dB),Min GSNR (dB),Max GSNR (dB)
0,Node 7 -> Node 8,551.3,9,90,6-QAM,21.728995,18.027611,25.778887
1,Node 7 -> Node 9,513.6,10,92,6-QAM,22.008039,18.654946,26.129178
2,Node 7 -> Node 11,109.5,2,96,6-QAM,27.293866,23.670129,30.790640
3,Node 7 -> Node 19,1126.9,21,93,5-QAM,18.794510,15.256620,22.942317
4,Node 7 -> Node 23,1027.0,19,90,5-QAM,19.159016,15.475739,23.358045
5,Node 7 -> Node 25,1445.4,26,100,4-QAM,17.420369,13.830842,21.779327
6,Node 7 -> Node 26,2086.8,34,93,3-QAM,14.870521,11.133248,19.563565
7,Node 7 -> Node 32,522.0,9,93,6-QAM,21.882316,18.383531,25.946003
8,Node 7 -> Node 35,105.0,2,92,6-QAM,27.774829,24.685066,31.104733
9,Node 7 -> Node 36,30.3,1,100,6-QAM,32.855520,31.872555,33.746646


### 5. Visualization: Path Performance & Heatmap

In [8]:
# Reach vs. GSNR Scatter Plot
fig_reach = go.Figure()
fig_reach.add_trace(go.Scatter(
    x=df_results["Distance (km)"],
    y=df_results["Mean GSNR (dB)"],
    mode="markers+text",
    marker=dict(size=12, color=df_results["Mean GSNR (dB)"], colorscale="Plasma", showscale=True),
    text=df_results["Connection"],
    textposition="top center"
))

fig_reach.update_layout(
    title="Lightpath Accumulated GSNR vs. Total Transmission Reach",
    xaxis_title="Physical Path Distance (km)",
    yaxis_title="Mean Path GSNR (dB)",
    template="plotly_white"
)
fig_reach.show()

# Heatmap Visualization using ModulationConnectionPlotter
plotter_mod = ModulationConnectionPlotter(modulation_connection=modulation_connection_matrix)
fig_mod = plotter_mod.plot(k_index=0, title="Primary Path (k=1) Modulation Format Matrix", matlab_indexing=False)
fig_mod.show()